In [1]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

Saving train.csv to train.csv


In [2]:
df = pd.read_csv("train.csv")

In [3]:
df.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


Информация о типах данных:

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


Число пропусков:

In [5]:
df.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


Средние значения, перцентили и т.д.:


In [6]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


Процент выживаемости по классам:

In [7]:
df.groupby('Pclass')['Survived'].mean()

,Survived
Pclass,
1,0.629630
2,0.472826
3,0.242363


Самое популярное мужское и женское имена:

In [11]:
import re

def parse_male_name(string):
  male_name = re.search(r'(Mr|Dr|Master|Sir|Rev|Don|Major|Col|Jonkheer|Capt)\. ([A-Za-z]+)', string)
  if male_name:
    return male_name.group(2)
  else:
    raise ValueError(f"{string} doesn't have supported male name prefix")

men_data = df[df['Sex'] == 'male']
men_names = men_data['Name'].apply(parse_male_name)
popular_men_name = men_names.value_counts().idxmax()
popular_men_name

'William'

In [12]:
def parse_female_name(string):
  female_name = re.search(r'(Mrs|Countess|Lady)\. \D*\(([A-Za-z]+)', string)  # women names are often in <husband's surname>, Mrs. ... (<name> ...) format
  if not female_name:
    female_name = re.search(r'(Miss|Mrs|Mme|Ms|Dr|Mlle)\. [^A-Za-z]*([A-Za-z]+)', string)
    if not female_name:
      raise ValueError(f"{string} doesn't have supported female name prefix")
  return female_name.group(2)

women_data = df[df['Sex'] == 'female']
women_names = women_data['Name'].apply(parse_female_name)
popular_women_name = women_names.value_counts().idxmax()
popular_women_name

'Anna'

самое популярное мужское и женское имя в каждом классе

In [13]:
from typing import Callable

def popular_gender_name_by_class(dataset: pd.DataFrame, sex: str, parse_func: Callable) -> pd.DataFrame:
  data_by_gender = df[df['Sex'] == sex]
  name_and_class_df = data_by_gender.loc[:, ['Name', 'Pclass']]
  name_and_class_df['Name'] = name_and_class_df['Name'].apply(parse_func)
  pop_name_answer = name_and_class_df.groupby('Pclass').value_counts()

  ans = pd.DataFrame(
                      columns=['Class', 'Most popular name'],
                      data = [[pclass, pop_name_answer[pclass].idxmax()] for pclass in range(1, 4)]
  )
  return ans


popular_gender_name_by_class(dataset=df, sex='female', parse_func=parse_female_name)

,Class,Most popular name
0,1,Elizabeth
1,2,Elizabeth
2,3,Anna


In [14]:
popular_gender_name_by_class(dataset=df, sex='male', parse_func=parse_male_name)

,Class,Most popular name
0,1,William
1,2,William
2,3,William


Пассажиры, которым больше 44 лет

In [15]:
df[df['Age'] > 44]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S
15,16,1,2,"Hewlett, Mrs. (Mary D Kingcome)",female,55.0,0,0,248706,16.0000,NaN,S
33,34,0,2,"Wheadon, Mr. Edward H",male,66.0,0,0,C.A. 24579,10.5000,NaN,S
52,53,1,1,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",female,49.0,1,0,PC 17572,76.7292,D33,C
...,...,...,...,...,...,...,...,...,...,...,...,...
857,858,1,1,"Daly, Mr. Peter Denis",male,51.0,0,0,113055,26.5500,E17,S
862,863,1,1,"Swift, Mrs. Frederick Joel (Margaret Welles Ba...",female,48.0,0,0,17466,25.9292,D17,S
871,872,1,1,"Beckwith, Mrs. Richard Leonard (Sallie Monypeny)",female,47.0,1,1,11751,52.5542,D35,S
873,874,0,3,"Vander Cruyssen, Mr. Victor",male,47.0,0,0,345765,9.0000,NaN,S


Пассажиры, которым меньше 44 лет и они мужского пола:

In [16]:
df[(df['Age'] < 44) & (df['Sex'] == 'male')]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.250,NaN,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.050,NaN,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.075,NaN,S
12,13,0,3,"Saundercock, Mr. William Henry",male,20.0,0,0,A/5. 2151,8.050,NaN,S
13,14,0,3,"Andersson, Mr. Anders Johan",male,39.0,1,5,347082,31.275,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
883,884,0,2,"Banfield, Mr. Frederick James",male,28.0,0,0,C.A./SOTON 34068,10.500,NaN,S
884,885,0,3,"Sutehall, Mr. Henry Jr",male,25.0,0,0,SOTON/OQ 392076,7.050,NaN,S
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.000,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.000,C148,C


Количество n-местных кабин

In [33]:
cabins = df['Cabin'].dropna().str.split().explode()
multi_cabins = cabins.value_counts().value_counts().drop(1)
multi_cabins = multi_cabins.rename_axis('Capacity').reset_index(name = 'cabins_number').sort_values(by='Capacity')
multi_cabins


,Capacity,cabins_number
0,2,44
2,3,6
1,4,7
